# 🔍 Smart Document Scanner + OCR
### Full Pipeline — Phase 1 → 4 | Google Colab + GitHub Edition

> **الصور والـ labels موجودة مباشرة في الريبو**  
> 1. Runtime → Change runtime type → **T4 GPU**  
> 2. Runtime → **Run all**

## ⚙️ Step 1 — Install dependencies

In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-fra tesseract-ocr-ara -q
!pip install pytesseract easyocr opencv-python-headless matplotlib numpy -q
print("✅ All packages installed")


## 📁 Step 2 — Clone repo & set working directory

In [ ]:
import os

REPO_URL  = "https://github.com/Aligfj/DocuScan_OCR.git"
REPO_NAME = "DocuScan_OCR"

if os.path.exists(REPO_NAME):
    !cd {REPO_NAME} && git pull --quiet
    print("✅ Repo updated")
else:
    !git clone --quiet {REPO_URL}
    print("✅ Repo cloned")

os.chdir(REPO_NAME)
print("📂 Working dir:", os.getcwd())


## 📂 Step 3 — Create output folders

In [ ]:
FOLDERS = [
    "dataset/images", "dataset/labels",
    "outputs/original", "outputs/gray", "outputs/blur",
    "outputs/edges", "outputs/contours", "outputs/warp",
    "outputs/clean", "outputs/ocr_results", "report",
]
for f in FOLDERS:
    os.makedirs(f, exist_ok=True)

print("✅ Folders ready")


## 🖼️ Step 4 — Discover images from repo

In [ ]:
import glob

image_paths = sorted(
    glob.glob("dataset/images/*.jpg") +
    glob.glob("dataset/images/*.jpeg") +
    glob.glob("dataset/images/*.png")
)

if not image_paths:
    print("⚠️  No images found in dataset/images/")
    print("    Add images to your repo under  dataset/images/test1.jpg  etc.")
else:
    print(f"✅ Found {len(image_paths)} image(s):")
    for p in image_paths:
        print(f"   {p}")


## 📦 Step 5 — Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110
import pytesseract
import easyocr
import time, csv
from difflib import SequenceMatcher

print("✅ Imports OK")
print("Tesseract:", pytesseract.get_tesseract_version())


## 🤖 Step 6 — Initialize EasyOCR

In [ ]:
import torch
use_gpu = torch.cuda.is_available()
print("GPU:", use_gpu)
reader = easyocr.Reader(['fr', 'en'], gpu=use_gpu)
print("✅ EasyOCR ready")


## 🛠️ Step 7 — Helper functions

In [ ]:
def show_pipeline(images, title="", save_path=None):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 3.2, 4))
    if n == 1: axes = [axes]
    fig.suptitle(title, fontsize=12, fontweight="bold")
    for ax, (label, img) in zip(axes, images.items()):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if len(img.shape)==3 else img,
                  cmap=None if len(img.shape)==3 else "gray")
        ax.set_title(label, fontsize=9); ax.axis("off")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

def order_points(pts):
    pts = pts.reshape(4, 2)
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]; rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]; rect[3] = pts[np.argmax(diff)]
    return rect

def find_document_contour(edges):
    cnts, _ = cv2.findContours(edges.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in sorted(cnts, key=cv2.contourArea, reverse=True):
        if cv2.contourArea(cnt) < 1000: continue
        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4: return approx
    return None

def warp_image(img, cnt):
    rect = order_points(cnt)
    tl, tr, br, bl = rect
    W = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
    H = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype="float32")
    return cv2.warpPerspective(img, cv2.getPerspectiveTransform(rect, dst), (W, H))

def preprocess_for_ocr(warped):
    g = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    g = cv2.equalizeHist(g)
    g = cv2.GaussianBlur(g, (3,3), 0)
    _, clean = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.medianBlur(clean, 3)

def cer(ref, hyp):
    if not ref: return None
    s = SequenceMatcher(None, ref, hyp)
    ed = len(ref)+len(hyp) - 2*sum(t.size for t in s.get_matching_blocks())
    return round(ed / max(len(ref),1), 4)

def word_accuracy(ref, hyp):
    if not ref: return None
    rw = ref.lower().split()
    return round(sum(1 for w in hyp.lower().split() if w in rw) / max(len(rw),1), 4)

print("✅ Helpers defined")


## 🔄 Step 8 — Full pipeline (Phase 1 + 2)

In [ ]:
def process_image(image_path):
    name = os.path.splitext(os.path.basename(image_path))[0]
    img  = cv2.imread(image_path)
    if img is None:
        print(f"  ⚠ Cannot read: {image_path}"); return None

    # CV pipeline
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 50, 150)

    doc_cnt  = find_document_contour(edges)
    img_draw = img.copy()
    warped = clean = None

    if doc_cnt is not None:
        cv2.drawContours(img_draw, [doc_cnt], -1, (0,255,0), 3)
        warped = warp_image(img, doc_cnt)
        clean  = preprocess_for_ocr(warped)
    else:
        print(f"  ⚠ {name}: no contour — OCR on full image")
        _, clean = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Tesseract
    t0 = time.time()
    tess_text = pytesseract.image_to_string(clean, lang="fra+eng", config="--psm 6")
    tess_time = round(time.time()-t0, 3)

    # EasyOCR
    t0 = time.time()
    easy_res      = reader.readtext(clean, detail=1)
    easy_time     = round(time.time()-t0, 3)
    easy_text     = "\n".join(t for _,t,_ in easy_res)
    easy_avg_conf = round(float(np.mean([c for _,_,c in easy_res])) if easy_res else 0, 3)

    # Save outputs
    for key, arr in [("original",img),("gray",gray),("blur",blur),
                     ("edges",edges),("contours",img_draw)]:
        cv2.imwrite(f"outputs/{key}/{name}.jpg", arr)
    if warped is not None: cv2.imwrite(f"outputs/warp/{name}.jpg",  warped)
    if clean  is not None: cv2.imwrite(f"outputs/clean/{name}.jpg", clean)
    with open(f"outputs/ocr_results/{name}_tesseract.txt","w",encoding="utf-8") as f:
        f.write(tess_text)
    with open(f"outputs/ocr_results/{name}_easyocr.txt","w",encoding="utf-8") as f:
        f.write(easy_text)

    return {
        "name": name, "img": img, "gray": gray, "blur": blur,
        "edges": edges, "contour_image": img_draw,
        "warped": warped, "clean": clean,
        "contour_found": doc_cnt is not None,
        "tess_text": tess_text.strip(), "tess_time": tess_time,
        "easy_text": easy_text.strip(), "easy_time": easy_time,
        "easy_avg_conf": easy_avg_conf,
    }


In [ ]:
results = []
for path in image_paths:
    print(f"⏳ {os.path.basename(path)} ...", end=" ")
    r = process_image(path)
    if r:
        results.append(r)
        print(f"{'✅' if r['contour_found'] else '⚠ no contour'}  "
              f"Tess:{r['tess_time']}s  Easy:{r['easy_time']}s")

print(f"\n✅ Done — {len(results)} image(s)")


## 📊 Step 9 — Pipeline panels

In [ ]:
for r in results:
    stages = {"Original": r["img"], "Gray": r["gray"],
              "Edges": r["edges"], "Contour": r["contour_image"]}
    if r["warped"] is not None: stages["Warped"] = r["warped"]
    if r["clean"]  is not None: stages["Clean"]  = r["clean"]
    show_pipeline(stages,
                  title=f"Pipeline — {r['name']}",
                  save_path=f"report/{r['name']}_panel.jpg")


## 📝 Step 10 — OCR text comparison

In [ ]:
for r in results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"OCR Comparison — {r['name']}", fontsize=13, fontweight="bold")

    if r["clean"] is not None:
        axes[0].imshow(r["clean"], cmap="gray")
    axes[0].set_title("Cleaned image"); axes[0].axis("off")

    for ax, txt, color, label in [
        (axes[1], r["tess_text"], "lightyellow", f"Tesseract ({r['tess_time']}s)"),
        (axes[2], r["easy_text"], "lightcyan",   f"EasyOCR ({r['easy_time']}s) conf:{r['easy_avg_conf']}")
    ]:
        ax.text(0.05, 0.95, txt[:500] or "(no result)",
                transform=ax.transAxes, fontsize=8,
                verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
        ax.set_title(label); ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"report/{r['name']}_ocr_compare.jpg", dpi=120, bbox_inches="tight")
    plt.show()


## ⚡ Step 11 — Speed comparison chart

In [ ]:
names = [r["name"] for r in results]
x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(max(8, len(names)*2), 5))
b1 = ax.bar(x-w/2, [r["tess_time"] for r in results], w, label="Tesseract", color="#4C72B0")
b2 = ax.bar(x+w/2, [r["easy_time"] for r in results], w, label="EasyOCR",   color="#DD8452")
ax.set_title("OCR Speed: Tesseract vs EasyOCR")
ax.set_xlabel("Image"); ax.set_ylabel("Seconds")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30)
ax.legend(); ax.bar_label(b1, fmt="%.2f"); ax.bar_label(b2, fmt="%.2f")
plt.tight_layout()
plt.savefig("report/speed_comparison.jpg", dpi=120); plt.show()


## 📏 Step 12 — Evaluation metrics (CER / Word Accuracy)

تأكد إن ملفات الـ labels موجودة في الريبو تحت `dataset/labels/test1.txt` ...  
إذا ما كانت موجودة، المقاييس ستظهر **N/A** وهذا طبيعي.

In [ ]:
eval_rows = []
for r in results:
    gt_path = f"dataset/labels/{r['name']}.txt"
    gt = open(gt_path, encoding="utf-8").read().strip() if os.path.exists(gt_path) else None
    eval_rows.append({
        "image":         r["name"],
        "contour_found": r["contour_found"],
        "ground_truth":  gt or "N/A",
        "tess_text":     r["tess_text"],
        "easy_text":     r["easy_text"],
        "tess_cer":      cer(gt, r["tess_text"])       if gt else "N/A",
        "easy_cer":      cer(gt, r["easy_text"])       if gt else "N/A",
        "tess_word_acc": word_accuracy(gt, r["tess_text"]) if gt else "N/A",
        "easy_word_acc": word_accuracy(gt, r["easy_text"]) if gt else "N/A",
        "tess_time_s":   r["tess_time"],
        "easy_time_s":   r["easy_time"],
        "easy_avg_conf": r["easy_avg_conf"],
    })

print(f"{'Image':<14} {'Contour':<9} {'Tess-CER':<11} {'Easy-CER':<11} {'Tess-WA':<10} {'Easy-WA':<10} {'Tess(s)':<9} {'Easy(s)'}")
print("-"*85)
for r in eval_rows:
    print(f"{r['image']:<14} {str(r['contour_found']):<9} "
          f"{str(r['tess_cer']):<11} {str(r['easy_cer']):<11} "
          f"{str(r['tess_word_acc']):<10} {str(r['easy_word_acc']):<10} "
          f"{r['tess_time_s']:<9} {r['easy_time_s']}")

with open("report/evaluation.csv","w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(eval_rows[0].keys()))
    w.writeheader(); w.writerows(eval_rows)
print("\n✅ report/evaluation.csv saved")


In [ ]:
labeled = [r for r in eval_rows if r["tess_cer"] != "N/A"]
if labeled:
    nl = [r["image"] for r in labeled]
    xl = np.arange(len(nl))
    fig, ax = plt.subplots(figsize=(max(8,len(nl)*2), 5))
    ax.bar(xl-0.2, [r["tess_cer"] for r in labeled], 0.4, label="Tesseract CER", color="#4C72B0")
    ax.bar(xl+0.2, [r["easy_cer"] for r in labeled], 0.4, label="EasyOCR CER",   color="#DD8452")
    ax.set_title("CER Comparison (lower = better)")
    ax.set_xticks(xl); ax.set_xticklabels(nl, rotation=30); ax.legend()
    plt.tight_layout()
    plt.savefig("report/cer_comparison.jpg", dpi=120); plt.show()
else:
    print("ℹ️  No labels found in dataset/labels/ → CER chart skipped")


## 🔄 Step 13 — Push outputs back to GitHub *(optional)*

غيّر `PUSH_TO_GITHUB = True` إذا تريد ترفع النتائج للريبو تلقائياً.  
ستحتاج تضع GitHub token في الخلية.

In [ ]:
PUSH_TO_GITHUB = False   # ← غيّر إلى True لتفعيل

if PUSH_TO_GITHUB:
    GH_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"   # ← ضع توكنك هنا
    REPO     = "Aligfj/DocuScan_OCR"

    !git config user.email "colab@colab.com"
    !git config user.name  "Colab"
    !git add outputs/ report/
    !git commit -m "Add pipeline outputs from Colab"
    !git push https://{GH_TOKEN}@github.com/{REPO}.git main
    print("✅ Pushed to GitHub")
else:
    print("Skipped — set PUSH_TO_GITHUB = True to enable")


## 📥 Step 14 — Download results as ZIP

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("DocuScan_results", "zip", ".", "report")
files.download("DocuScan_results.zip")
print("✅ Download started")


## ✅ Step 15 — Completion checklist

In [ ]:
labeled_count = len([r for r in eval_rows if r["tess_cer"] != "N/A"])

checks = [
    ("Packages installed",                          True),
    ("Repo cloned from GitHub",                     True),
    ("Folder structure created",                    True),
    ("Images loaded from repo",                     len(image_paths) > 0),
    ("Grayscale + Blur + Canny applied",            True),
    ("Document contour detection",                  True),
    ("Perspective correction",                      True),
    ("Otsu thresholding for clean image",           True),
    ("Tesseract OCR saved per image",               True),
    ("EasyOCR saved with confidence scores",        True),
    ("Pipeline visual panels saved",                len(results) >= 1),
    ("OCR text comparison figures saved",           True),
    ("Speed chart saved",                           True),
    ("evaluation.csv saved",                        os.path.exists("report/evaluation.csv")),
    ("CER / WA metrics (needs labels)",             labeled_count > 0),
]

print("  COMPLETION CHECKLIST")
print("  " + "="*48)
for desc, ok in checks:
    print(f"  {'✅' if ok else '❌'}  {desc}")
print(f"\n  {sum(ok for _,ok in checks)}/{len(checks)} checks passed")
